In [133]:
import pandas as pd
import numpy as np
import requests
import os
import datetime as dt
import re
import ast

pd.set_option('display.max_columns', 999)

In [65]:
#Helper Functions

def convert_to_num(df):
    for col in df.columns:
        try:
            pd.to_numeric(df[col])
        except:
            print(f"Column {col} cannot be converted to numeric")
            continue

def labelDom(row):
    if row['DOM'] < 14:
        return True
    else:
        return False

def binaryDom(df):
    df['binaryDOM'] = df.apply(lambda row: labelDom(row), axis = 1)

def createDOM(df: pd.DataFrame, timestamp = False, convert = False):
    dom = df['soldDate'] - df['listDate']
    df['DOM'] = dom.astype("string")
    df.dropna(subset=['DOM'], inplace = True)
    df['DOM'] = df['DOM'].str.slice_replace(start=2)
    df['DOM'] = df['DOM'].astype("int")

def dfRemoveNA(df):
    df = df.replace('', np.nan)
    if sum(df.isna().sum()) != 0:
        df = df.dropna()
    return df

def dfFillNA(df, column):
    df = df.replace('', np.nan)
    df = df.fillna(value = {column: "No "+column})
    return df

def dfExtractFeatures(df: pd.DataFrame, key: str, columns: list):
    if type(df[key].values[0]) == str:
        serie = df[key].apply(lambda x: ast.literal_eval(str(x)))
        d = pd.DataFrame.from_dict(serie.tolist())
    elif type(df[key].values[0]) == dict:
        serie = pd.Series.to_dict(df[key])
        d = pd.DataFrame.from_dict(serie, orient='index')
    for i in range(len(columns)):
        df[columns[i]] = d[columns[i]]

    return df

def dfPriceDifference(df: pd.DataFrame):
    #should only be for clustering purposes only
    df['PriceDifference'] = df['soldPrice'] - df['listPrice']

def dfDatesRemoveTimeStamp(df: pd.DataFrame):
    df['listDate'] = df['listDate'].str.slice_replace(start = 10, repl="")
    df['soldDate'] = df['soldDate'].str.slice_replace(start = 10, repl="")

def dfConvertToDatetime(df: pd.DataFrame, timestamp = False):
    if type(df['listDate'].values[0]) == str:
        dfDatesRemoveTimeStamp(df)
    df['listDate'] = pd.to_datetime(df['listDate'])
    df['soldDate'] = pd.to_datetime(df['soldDate'])

def removeFeatures(df: pd.DataFrame, columns: list):
    #df = df.drop(['images', 'resource', 'timestamps','permissions','lastStatus','status','boardId','agents'], axis = 1)
    df = df.drop(columns, axis = 1)
    return df

#Data Augmentation

def countLevels(df: pd.DataFrame, column):
    series = df[column].value_counts()
    return series

def augmentCategorical(df, column, percent = 0.04):
    series = countLevels(df, column)
    total = np.sum(series.values)
    percentage = [x/total for x in series.values]
    
    for i in range(len(percentage)):
        if percentage[i] > percent:
            pass
        else:
            df = df.replace(to_replace = {column: series.index[i]}, value = 'Other '+column)

    return df

def augmentType(df, type, columns):
    for col in columns:
        df[col] = df[col].astype(df[col].astype(type))

    return df

def augmentNumericals(df):
    for i in range(len(df.columns)):
        if (type(df[df.columns[i]].values[0]) == str):
            numerical = re.search('^[-+]?[0-9]*\.?[0-9]+$', df[df.columns[i]].values[0])
            if numerical is not None:
                df[df.columns[i]] = df[df.columns[i]].astype(float)
    return df

def removeLowLevels(df, column):
    lst = []
    for i in range(len(df[column].value_counts().index)):
        if df[column].value_counts().values[i] < 11:
            lst.append(df[column].value_counts().index[i])

    for i in range(len(lst)):
        df = df[df[column] != lst[i]]
    return df

def removeOutliers(df):
    percentile90 = df['soldPrice'].quantile(0.90)
    percentile10 = df['soldPrice'].quantile(0.10)

    iqr = percentile90 - percentile10

    upper_limit = percentile90 + 3 * iqr
    lower_limit = percentile10 - 3 * iqr

    # df[df['soldPrice'] > upper_limit]
    # df[df['soldPrice'] < lower_limit]

    df = df[df['soldPrice'] < upper_limit]
    df = df[df['soldPrice'] > lower_limit]

    return df


In [85]:
raw_lease_df = pd.read_csv(os.getcwd()+"\\data\\raw_lease_data.csv")

In [12]:
raw_lease_df.head()

,listDate,images,address,soldDate,resource,timestamps,type,mlsNumber,permissions,soldPrice,...,city,numBathrooms,numBedrooms,sqft,style,district,neighborhood,yearBuilt,latitude,longitude
0,2019-04-16T00:00:00.000Z,"['IMG-N4418450_1.jpg', 'IMG-N4418450_2.jpg', '...","{'area': 'York', 'zip': 'L3Y0E1', 'country': N...",2019-05-06T00:00:00.000Z,Property:2381,{'listingUpdated': '2019-05-08T10:56:42.000Z'},Lease,N4418450,"{'displayAddressOnInternet': 'Y', 'displayPubl...",2200.0,...,Newmarket,4,4,1500-2000,3-Storey,Newmarket,Glenway Estates,0-5,44.05193999999999,-79.49083999999999
1,2019-05-01T00:00:00.000Z,"['IMG-N4435010_1.jpg', 'IMG-N4435010_2.jpg', '...","{'area': 'York', 'zip': 'L4H1T3', 'country': N...",2019-05-06T00:00:00.000Z,Property:2381,{'listingUpdated': '2019-05-08T15:58:16.000Z'},Lease,N4435010,"{'displayAddressOnInternet': 'Y', 'displayPubl...",1350.0,...,Vaughan,1,2,,2-Storey,Vaughan,Sonoma Heights,16-30,43.8168699,-79.618265
2,2019-05-03T00:00:00.000Z,"['IMG-N4437661_1.jpg', 'IMG-N4437661_2.jpg', '...","{'area': 'York', 'zip': 'L4E0T9', 'country': N...",2019-05-06T00:00:00.000Z,Property:2381,{'listingUpdated': '2019-05-07T11:06:02.000Z'},Lease,N4437661,"{'displayAddressOnInternet': 'Y', 'displayPubl...",3000.0,...,Richmond Hill,4,5,3000-3500,2-Storey,Richmond Hill,Jefferson,0-5,43.9175319,-79.43661039999999
3,2019-04-03T00:00:00.000Z,"['IMG-N4405561_1.jpg', 'IMG-N4405561_2.jpg', '...","{'area': 'York', 'zip': 'L9N 0T3', 'country': ...",2019-05-06T00:00:00.000Z,Property:2381,{'listingUpdated': '2019-05-08T14:23:19.000Z'},Lease,N4405561,"{'displayAddressOnInternet': 'Y', 'displayPubl...",2400.0,...,East Gwillimbury,4,4,3000-3500,2-Storey,East Gwillimbury,Holland Landing,New,44.09185,-79.50193
4,2019-04-24T00:00:00.000Z,['IMG-W4426017_1.jpg'],"{'area': 'Peel', 'zip': 'L4X4H2', 'country': N...",2019-05-06T00:00:00.000Z,Property:2381,{'listingUpdated': '2019-05-09T13:23:15.000Z'},Lease,W4426017,"{'displayAddressOnInternet': 'Y', 'displayPubl...",3400.0,...,Mississauga,4,4,,2-Storey,Mississauga,Hurontario,,43.6064424,-79.6422499


The datatypes are mainly objects, but there are dictionaries and floats/integers that are located inside each objects that may need to be converted. Certain predictors like number of bathrooms can be left as strings, 

In [13]:
raw_lease_df.dtypes

listDate         object
images           object
address          object
soldDate         object
resource         object
timestamps       object
type             object
mlsNumber        object
permissions      object
soldPrice       float64
details          object
class            object
map              object
listPrice       float64
lastStatus       object
status           object
boardId           int64
agents           object
area             object
city             object
numBathrooms     object
numBedrooms      object
sqft             object
style            object
district         object
neighborhood     object
yearBuilt        object
latitude         object
longitude        object
dtype: object

In [119]:
address = ['area', 'city', 'district', 'neighborhood']
details = ['numBathrooms','numBedrooms','sqft','style',]
map_ = ['latitude', 'longitude']
columns = ['listDate','soldDate','address','details','images', 'resource', 'timestamps','permissions','lastStatus','status','boardId','agents','type','mlsNumber','map']

lease_df = dfExtractFeatures(raw_lease_df, key = 'address', columns = address)
lease_df = dfExtractFeatures(lease_df, key = 'details', columns = details)
lease_df = dfExtractFeatures(lease_df, key = 'map', columns = map_)

In [134]:
lease_df.head()

,listDate,images,address,soldDate,resource,timestamps,type,mlsNumber,permissions,soldPrice,details,class,map,listPrice,lastStatus,status,boardId,agents,area,city,district,neighborhood,numBathrooms,numBedrooms,sqft,style,latitude,longitude,DOM,avg_sqft,ppsqft
0,2019-04-16,"['IMG-N4418450_1.jpg', 'IMG-N4418450_2.jpg', '...","{'area': 'York', 'zip': 'L3Y0E1', 'country': N...",2019-05-06,Property:2381,{'listingUpdated': '2019-05-08T10:56:42.000Z'},Lease,N4418450,"{'displayAddressOnInternet': 'Y', 'displayPubl...",2200.0,"{'numBathrooms': '4', 'numBedrooms': '4', 'sqf...",ResidentialProperty,"{'latitude': '44.05193999999999', 'longitude':...",2100.0,Lsd,U,2,[],York,Newmarket,Newmarket,Glenway Estates,4,4,1500-2000,3-Storey,44.05193999999999,-79.49083999999999,20,1750.0,1.200000
1,2019-05-01,"['IMG-N4435010_1.jpg', 'IMG-N4435010_2.jpg', '...","{'area': 'York', 'zip': 'L4H1T3', 'country': N...",2019-05-06,Property:2381,{'listingUpdated': '2019-05-08T15:58:16.000Z'},Lease,N4435010,"{'displayAddressOnInternet': 'Y', 'displayPubl...",1350.0,"{'numBathrooms': '1', 'numBedrooms': '2', 'sqf...",ResidentialProperty,"{'latitude': '43.8168699', 'longitude': '-79.6...",1350.0,Lsd,U,2,[],York,Vaughan,Vaughan,Sonoma Heights,1,2,NaN,2-Storey,43.8168699,-79.618265,5,NaN,NaN
2,2019-05-03,"['IMG-N4437661_1.jpg', 'IMG-N4437661_2.jpg', '...","{'area': 'York', 'zip': 'L4E0T9', 'country': N...",2019-05-06,Property:2381,{'listingUpdated': '2019-05-07T11:06:02.000Z'},Lease,N4437661,"{'displayAddressOnInternet': 'Y', 'displayPubl...",3000.0,"{'numBathrooms': '4', 'numBedrooms': '5', 'sqf...",ResidentialProperty,"{'latitude': '43.9175319', 'longitude': '-79.4...",2999.0,Lsd,U,2,[],York,Richmond Hill,Richmond Hill,Jefferson,4,5,3000-3500,2-Storey,43.9175319,-79.43661039999999,3,3250.0,0.922769
3,2019-04-03,"['IMG-N4405561_1.jpg', 'IMG-N4405561_2.jpg', '...","{'area': 'York', 'zip': 'L9N 0T3', 'country': ...",2019-05-06,Property:2381,{'listingUpdated': '2019-05-08T14:23:19.000Z'},Lease,N4405561,"{'displayAddressOnInternet': 'Y', 'displayPubl...",2400.0,"{'numBathrooms': '4', 'numBedrooms': '4', 'sqf...",ResidentialProperty,"{'latitude': '44.09185', 'longitude': '-79.501...",2400.0,Lsd,U,2,[],York,East Gwillimbury,East Gwillimbury,Holland Landing,4,4,3000-3500,2-Storey,44.09185,-79.50193,33,3250.0,0.738462
4,2019-04-24,['IMG-W4426017_1.jpg'],"{'area': 'Peel', 'zip': 'L4X4H2', 'country': N...",2019-05-06,Property:2381,{'listingUpdated': '2019-05-09T13:23:15.000Z'},Lease,W4426017,"{'displayAddressOnInternet': 'Y', 'displayPubl...",3400.0,"{'numBathrooms': '4', 'numBedrooms': '4', 'sqf...",ResidentialProperty,"{'latitude': '43.6064424', 'longitude': '-79.6...",3400.0,Lsd,U,2,[],Peel,Mississauga,Mississauga,Hurontario,4,4,NaN,2-Storey,43.6064424,-79.6422499,12,NaN,NaN


In [120]:
lease_df.columns

Index(['listDate', 'images', 'address', 'soldDate', 'resource', 'timestamps',
       'type', 'mlsNumber', 'permissions', 'soldPrice', 'details', 'class',
       'map', 'listPrice', 'lastStatus', 'status', 'boardId', 'agents', 'area',
       'city', 'district', 'neighborhood', 'numBathrooms', 'numBedrooms',
       'sqft', 'style', 'latitude', 'longitude', 'DOM', 'avg_sqft'],
      dtype='object')

Feature Engineering

In [121]:
pd.unique(lease_df['sqft'].values)

array(['1500-2000', '', '3000-3500', '2500-3000', '700-1100', '2000-2500',
       '3500-5000', '600-699', '1100-1500', '1200-1399', '500-599',
       '900-999', '800-899', '1400-1599', '700-799', '1000-1199',
       '1600-1799', '2000-2249', '0-499', '1800-1999', '< 700',
       '2250-2499', '5000+', '3500-3749', '500-699', '2750-2999',
       '3250-3499', '4000-4249', '2500-2749', '3000-3249', '1100-1299',
       '700-899', '3750-3999', '1300-1499', '900-1099', '4500-4749',
       '4750-4999', '4250-4499', nan], dtype=object)

In [122]:
lease_df.replace({"": np.nan}, inplace = True)
lease_df.replace({float("nan"): np.nan}, inplace = True)

,listDate,images,address,soldDate,resource,timestamps,type,mlsNumber,permissions,soldPrice,...,district,neighborhood,numBathrooms,numBedrooms,sqft,style,latitude,longitude,DOM,avg_sqft
0,2019-04-16,"['IMG-N4418450_1.jpg', 'IMG-N4418450_2.jpg', '...","{'area': 'York', 'zip': 'L3Y0E1', 'country': N...",2019-05-06,Property:2381,{'listingUpdated': '2019-05-08T10:56:42.000Z'},Lease,N4418450,"{'displayAddressOnInternet': 'Y', 'displayPubl...",2200.0,...,Newmarket,Glenway Estates,4,4,1500-2000,3-Storey,44.05193999999999,-79.49083999999999,20,1750.0
1,2019-05-01,"['IMG-N4435010_1.jpg', 'IMG-N4435010_2.jpg', '...","{'area': 'York', 'zip': 'L4H1T3', 'country': N...",2019-05-06,Property:2381,{'listingUpdated': '2019-05-08T15:58:16.000Z'},Lease,N4435010,"{'displayAddressOnInternet': 'Y', 'displayPubl...",1350.0,...,Vaughan,Sonoma Heights,1,2,NaN,2-Storey,43.8168699,-79.618265,5,NaN
2,2019-05-03,"['IMG-N4437661_1.jpg', 'IMG-N4437661_2.jpg', '...","{'area': 'York', 'zip': 'L4E0T9', 'country': N...",2019-05-06,Property:2381,{'listingUpdated': '2019-05-07T11:06:02.000Z'},Lease,N4437661,"{'displayAddressOnInternet': 'Y', 'displayPubl...",3000.0,...,Richmond Hill,Jefferson,4,5,3000-3500,2-Storey,43.9175319,-79.43661039999999,3,3250.0
3,2019-04-03,"['IMG-N4405561_1.jpg', 'IMG-N4405561_2.jpg', '...","{'area': 'York', 'zip': 'L9N 0T3', 'country': ...",2019-05-06,Property:2381,{'listingUpdated': '2019-05-08T14:23:19.000Z'},Lease,N4405561,"{'displayAddressOnInternet': 'Y', 'displayPubl...",2400.0,...,East Gwillimbury,Holland Landing,4,4,3000-3500,2-Storey,44.09185,-79.50193,33,3250.0
4,2019-04-24,['IMG-W4426017_1.jpg'],"{'area': 'Peel', 'zip': 'L4X4H2', 'country': N...",2019-05-06,Property:2381,{'listingUpdated': '2019-05-09T13:23:15.000Z'},Lease,W4426017,"{'displayAddressOnInternet': 'Y', 'displayPubl...",3400.0,...,Mississauga,Hurontario,4,4,NaN,2-Storey,43.6064424,-79.6422499,12,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
320766,2023-03-14,"['IMG-W5965967_1.jpg', 'IMG-W5965967_2.jpg', '...","{'area': 'Toronto', 'zip': 'M6S 1A1', 'country...",2023-04-18,Property:2381,{'listingUpdated': '2023-04-19T21:21:01.000Z'},Lease,W5965967,"{'displayAddressOnInternet': 'Y', 'displayPubl...",4500.0,...,Toronto W02,High Park North,1,2,NaN,2-Storey,43.6607634,-79.4601019,35,949.5
320767,2023-03-23,"['IMG-W5985741_1.jpg', 'IMG-W5985741_2.jpg', '...","{'area': 'Toronto', 'zip': 'M6P2G3', 'country'...",2023-04-18,Property:2381,{'listingUpdated': '2023-04-19T17:01:48.000Z'},Lease,W5985741,"{'displayAddressOnInternet': 'Y', 'displayPubl...",3700.0,...,Toronto C15,Henry Farm,1,1,500-599,Apartment,43.7716467,-79.3446635,26,NaN
320768,2023-03-20,"['IMG-C5975287_1.jpg', 'IMG-C5975287_2.jpg', '...","{'area': 'Toronto', 'zip': 'M2J1M5', 'country'...",2023-04-18,Property:2381,{'listingUpdated': '2023-04-19T22:13:37.000Z'},Lease,C5975287,"{'displayAddressOnInternet': 'Y', 'displayPubl...",2500.0,...,Toronto C01,Bay Street Corridor,1,1,600-699,Apartment,43.6505301,-79.3821482,29,549.5
320769,2023-04-13,"['IMG-C6024761_1.jpg', 'IMG-C6024761_2.jpg', '...","{'area': 'Toronto', 'zip': 'M5H0B1', 'country'...",2023-04-18,Property:2381,{'listingUpdated': '2023-04-19T22:49:33.000Z'},Lease,C6024761,"{'displayAddressOnInternet': 'Y', 'displayPubl...",3500.0,...,Barrie,North Shore,1,1,700-799,Apartment,44.3896491,-79.68452479999999,5,649.5


In [123]:
dfConvertToDatetime(lease_df)

In [124]:
createDOM(lease_df)

In [125]:
lease_df['avg_sqft'] = [(int(value.split("-")[0])+int(value.split("-")[1]))/2 if not isinstance(value, float) and '-' in value else np.nan for value in lease_df['sqft']]

In [126]:
lease_df['avg_sqft']

0         1750.0
1            NaN
2         3250.0
3         3250.0
4            NaN
           ...  
320766       NaN
320767     549.5
320768     649.5
320769     749.5
320770       NaN
Name: avg_sqft, Length: 320770, dtype: float64

In [127]:
lease_df['ppsqft'] = lease_df['listPrice']/lease_df['avg_sqft']

In [130]:
lease_df['ppsqft']

0         1.200000
1              NaN
2         0.922769
3         0.738462
4              NaN
            ...   
320766         NaN
320767    6.460419
320768    3.849115
320769    4.269513
320770         NaN
Name: ppsqft, Length: 320770, dtype: float64

In [141]:
pd.unique(lease_df['numBedrooms'].values)

array(['4', '2', '5', '3', '1', '0', '6', nan, '7', '8'], dtype=object)

In [142]:
lease_df['bathtobed_ratio'] = lease_df['numBathrooms'].astype("Int64")/lease_df['numBedrooms'].astype("Int64")

In [150]:
lease_df['latitude'] = lease_df['latitude'].astype("float")
lease_df['longitude'] = lease_df['longitude'].astype("float")

In [197]:
lease_df.head()

,listDate,images,address,soldDate,resource,timestamps,type,mlsNumber,permissions,soldPrice,details,class,map,listPrice,lastStatus,status,boardId,agents,area,city,district,neighborhood,numBathrooms,numBedrooms,sqft,style,latitude,longitude,DOM,avg_sqft,ppsqft,bathtobed_ratio
0,2019-04-16,"['IMG-N4418450_1.jpg', 'IMG-N4418450_2.jpg', '...","{'area': 'York', 'zip': 'L3Y0E1', 'country': N...",2019-05-06,Property:2381,{'listingUpdated': '2019-05-08T10:56:42.000Z'},Lease,N4418450,"{'displayAddressOnInternet': 'Y', 'displayPubl...",2200.0,"{'numBathrooms': '4', 'numBedrooms': '4', 'sqf...",ResidentialProperty,"{'latitude': '44.05193999999999', 'longitude':...",2100.0,Lsd,U,2,[],York,Newmarket,Newmarket,Glenway Estates,4,4,1500-2000,3-Storey,44.051940,-79.490840,20,1750.0,1.200000,1.0
1,2019-05-01,"['IMG-N4435010_1.jpg', 'IMG-N4435010_2.jpg', '...","{'area': 'York', 'zip': 'L4H1T3', 'country': N...",2019-05-06,Property:2381,{'listingUpdated': '2019-05-08T15:58:16.000Z'},Lease,N4435010,"{'displayAddressOnInternet': 'Y', 'displayPubl...",1350.0,"{'numBathrooms': '1', 'numBedrooms': '2', 'sqf...",ResidentialProperty,"{'latitude': '43.8168699', 'longitude': '-79.6...",1350.0,Lsd,U,2,[],York,Vaughan,Vaughan,Sonoma Heights,1,2,NaN,2-Storey,43.816870,-79.618265,5,NaN,NaN,0.5
2,2019-05-03,"['IMG-N4437661_1.jpg', 'IMG-N4437661_2.jpg', '...","{'area': 'York', 'zip': 'L4E0T9', 'country': N...",2019-05-06,Property:2381,{'listingUpdated': '2019-05-07T11:06:02.000Z'},Lease,N4437661,"{'displayAddressOnInternet': 'Y', 'displayPubl...",3000.0,"{'numBathrooms': '4', 'numBedrooms': '5', 'sqf...",ResidentialProperty,"{'latitude': '43.9175319', 'longitude': '-79.4...",2999.0,Lsd,U,2,[],York,Richmond Hill,Richmond Hill,Jefferson,4,5,3000-3500,2-Storey,43.917532,-79.436610,3,3250.0,0.922769,0.8
3,2019-04-03,"['IMG-N4405561_1.jpg', 'IMG-N4405561_2.jpg', '...","{'area': 'York', 'zip': 'L9N 0T3', 'country': ...",2019-05-06,Property:2381,{'listingUpdated': '2019-05-08T14:23:19.000Z'},Lease,N4405561,"{'displayAddressOnInternet': 'Y', 'displayPubl...",2400.0,"{'numBathrooms': '4', 'numBedrooms': '4', 'sqf...",ResidentialProperty,"{'latitude': '44.09185', 'longitude': '-79.501...",2400.0,Lsd,U,2,[],York,East Gwillimbury,East Gwillimbury,Holland Landing,4,4,3000-3500,2-Storey,44.091850,-79.501930,33,3250.0,0.738462,1.0
4,2019-04-24,['IMG-W4426017_1.jpg'],"{'area': 'Peel', 'zip': 'L4X4H2', 'country': N...",2019-05-06,Property:2381,{'listingUpdated': '2019-05-09T13:23:15.000Z'},Lease,W4426017,"{'displayAddressOnInternet': 'Y', 'displayPubl...",3400.0,"{'numBathrooms': '4', 'numBedrooms': '4', 'sqf...",ResidentialProperty,"{'latitude': '43.6064424', 'longitude': '-79.6...",3400.0,Lsd,U,2,[],Peel,Mississauga,Mississauga,Hurontario,4,4,NaN,2-Storey,43.606442,-79.642250,12,NaN,NaN,1.0


In [144]:
for i in range(10):
    lat, long = lease_df['latitude'].values[i]+ 0.005, lease_df['latitude'].values[i] - 0.005, lease_df['latitude'].values[i]
    print(lease_df['latitude'].values[i], lease_df['longitude'].values[i])

    print()

44.05193999999999 -79.49083999999999
43.8168699 -79.618265
43.9175319 -79.43661039999999
44.09185 -79.50193
43.6064424 -79.6422499
43.7468834 -79.3951895
43.4981375 -79.7220875
43.3450861 -79.7721256
44.0952303 -79.5717308
43.82614600000001 -79.4073715


In [159]:
len(lease_df)

320770

In [160]:
lease_df.index

Int64Index([     0,      1,      2,      3,      4,      5,      6,      7,
                 8,      9,
            ...
            320761, 320762, 320763, 320764, 320765, 320766, 320767, 320768,
            320769, 320770],
           dtype='int64', length=320770)

In [195]:
def get_radius(lat, long, value = None):
    if value:
        return (lat + value, lat - value, long + value, long - value)
    else:
        return (lat + 0.005, lat - 0.005, long + 0.005, long - 0.005)

def avg_radius(df):
    avg_prices = []
    for i in range(len(df)):
        avg_price = []
        radius = get_radius(df['latitude'].values[i], df['longitude'].values[i])
        for j in range(len(df)):
            if j != i:
                if (radius[0] <= df['latitude'].values[j] <= radius[1]) and (radius[2] <=  df['longitude'].values[j] <= radius[3]):
                    avg_price.append(df['listPrice'].values[j])
        if len(avg_price) == 0:
            avg_prices.append(np.nan)
        else:
            avg_prices.append(np.mean(avg_price))

    print(avg_prices)

    return avg_prices

def get_radius_(mlsNumber, api_key):
    url = f"https://api.repliers.io/listings/{mlsNumber}?similar?radius=1"

    payload = {}
    headers = {'repliers-api-key': api_key}
    r = requests.request("GET",url, params=payload, headers=headers)
    data = r.json()
    df = pd.DataFrame(data['listings'])
    return df

In [196]:
a = get_radius(lease_df['latitude'].values[0],lease_df['longitude'].values[0])

df = get_radius_("N4418450", "v9iud7CVdiWNLF94rT6nwvIPfKuXN0")

KeyError: 'listings'

In [203]:
for i in range(1000):
    url = f"https://api.repliers.io/listings/{lease_df['mlsNumber'].values[i]}"
    payload = {}
    headers = {'repliers-api-key': "v9iud7CVdiWNLF94rT6nwvIPfKuXN0"}
    r = requests.request("GET",url, params=payload, headers=headers)
    data = r.json()
    print(data["comparables"])

[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [205]:
url = f"https://api.repliers.io/listings/N6039627"
payload = {}
headers = {'repliers-api-key': "v9iud7CVdiWNLF94rT6nwvIPfKuXN0"}
r = requests.request("GET",url, params=payload, headers=headers)
data = r.json()
print(data["comparables"])

[{'listDate': '2023-04-24T00:00:00.000Z', 'images': ['IMG-N6044945_1.jpg', 'IMG-N6044945_2.jpg', 'IMG-N6044945_3.jpg', 'IMG-N6044945_4.jpg', 'IMG-N6044945_5.jpg', 'IMG-N6044945_6.jpg', 'IMG-N6044945_7.jpg', 'IMG-N6044945_8.jpg', 'IMG-N6044945_9.jpg', 'IMG-N6044945_10.jpg', 'IMG-N6044945_11.jpg', 'IMG-N6044945_12.jpg', 'IMG-N6044945_13.jpg', 'IMG-N6044945_14.jpg', 'IMG-N6044945_15.jpg', 'IMG-N6044945_16.jpg', 'IMG-N6044945_17.jpg', 'IMG-N6044945_18.jpg', 'IMG-N6044945_19.jpg', 'IMG-N6044945_20.jpg', 'IMG-N6044945_21.jpg', 'IMG-N6044945_22.jpg', 'IMG-N6044945_23.jpg', 'IMG-N6044945_24.jpg', 'IMG-N6044945_25.jpg', 'IMG-N6044945_26.jpg', 'IMG-N6044945_27.jpg', 'IMG-N6044945_28.jpg', 'IMG-N6044945_29.jpg', 'IMG-N6044945_30.jpg', 'IMG-N6044945_31.jpg', 'IMG-N6044945_32.jpg', 'IMG-N6044945_33.jpg', 'IMG-N6044945_34.jpg', 'IMG-N6044945_35.jpg', 'IMG-N6044945_36.jpg', 'IMG-N6044945_37.jpg', 'IMG-N6044945_38.jpg', 'IMG-N6044945_39.jpg', 'IMG-N6044945_40.jpg'], 'address': {'area': 'York', 'zip': 